### 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript)

In [9]:
language = "pt"

In [10]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""
def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio_wav'
  with open(file_name + '.wav', 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=True))

Ouvindo...



<IPython.core.display.Javascript object>

### 2. Reconhecimento de Fala com Whisper (OpenAI)

In [11]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)


 Alô, alô, alô, alô, alô, som, som, som, som, som.


### 3. Integração com a API do ChatGPT

In [13]:
!pip install openai

In [14]:
from openai import OpenAI

client = OpenAI(api_key="SUA API KEy")

response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [ { "role": "user", "content": transcription } ]
)

chatgpt_response = response.choices[0].message.content
print(chatgpt_response)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS)

In [ ]:
!pip install gTTS

In [ ]:
from gtts import gTTS

gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)
response_audio = "/content/response_audio.wav"
gtts_object.save()
display(Audio(response_audio, autoplay=True))